In [ ]:
!pip install -q transformers datasets evaluate accelerate peft bitsandbytes safetensors --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 16.1 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForSequenceClassification
from transformers import Trainer
from peft import IA3Config

In [ ]:
import transformers
print(transformers.__version__)

4.57.1


In [ ]:
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
from datasets import load_dataset
yelpDataset = load_dataset("yelp_polarity")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

In [ ]:
modelName = "bert-base-uncased"

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(modelName)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

yelpDataset = yelpDataset.map(tokenize, batched = True)
yelpDataset = yelpDataset.rename_column("label", "labels")
yelpDataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/560000 [00:00<?, ? examples/s]

Map:   0%|          | 0/38000 [00:00<?, ? examples/s]

In [ ]:
train_data = yelpDataset["train"].shuffle(seed=42).select(range(10000))
eval_data  = yelpDataset["test"].shuffle(seed=42).select(range(4000))

#Accuracy Function

In [ ]:
from sklearn.metrics import accuracy_score
def accuracyFunction(model, dataset, batch_size=32):
    model.eval()
    preds, labels = [], []
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    return accuracy_score(labels, preds)

#LoRa

In [ ]:
LoRaModel = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).to(device)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "key", "value", "dense"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
LoRaModel = get_peft_model(LoRaModel, lora_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import TrainingArguments
LoRa_args = TrainingArguments(
    eval_strategy="epoch",
    learning_rate=5e-5,
    save_strategy="no",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    report_to="none"
)

In [ ]:
trainer_lora = Trainer(
    model=LoRaModel,
    args=LoRa_args,
    train_dataset=train_data,
    eval_dataset=eval_data
)

In [ ]:
if torch.cuda.is_available():
  torch.cuda.empty_cache()
  memoryBeforeLora = torch.cuda.memory_allocated()

else:
  memoryBeforeLora = 0

In [ ]:
import time
start = time.time()
trainer_lora.train()
end = time.time()

Epoch,Training Loss,Validation Loss
1,0.267300,0.227382
2,0.306700,0.223982
3,0.155800,0.219340


In [ ]:
LoRaTrainingTime = end - start

In [ ]:
if torch.cuda.is_available():
    memoryAfterLoRa = torch.cuda.memory_allocated()
else:
    memoryAfterLoRa = 0

In [ ]:
LoRaAccuracy = accuracyFunction(LoRaModel, eval_data)

#Q_LoRa

In [ ]:
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

QLoRaModel = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    quantization_config=bnb_config,
    num_labels=2,
    device_map="auto"
)

QLoRaModel = prepare_model_for_kbit_training(QLoRaModel)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import LoraConfig
QLoRa_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query","key","value","output.dense","intermediate.dense"],
    bias="none",
    task_type="SEQ_CLS"
)


In [ ]:
from peft import get_peft_model
QLoRaModel = get_peft_model(QLoRaModel, QLoRa_config)

In [ ]:
from transformers import TrainingArguments
QLoRa_args = TrainingArguments(
    eval_strategy="epoch",
    learning_rate=5e-5,
    save_strategy="no",
    per_device_train_batch_size=16,
    num_train_epochs=1,
    fp16=False,
    bf16=False,
    logging_steps=10,
    report_to="none"
)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    memoryBeforeQLora = torch.cuda.memory_allocated()
else:
    memoryBeforeQLora = 0

In [ ]:
from transformers import Trainer
trainer_qlora = Trainer(
    model=QLoRaModel,
    args=QLoRa_args,
    train_dataset=train_data,
    eval_dataset=eval_data
)


In [ ]:
import time
start = time.time()
trainer_qlora.train()
end = time.time()

QLoRaTrainingTime = end - start

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.285800,0.229832


In [ ]:
if torch.cuda.is_available():
    memoryAfterQLoRa = torch.cuda.memory_allocated()
else:
    memoryAfterQLoRa = 0

In [ ]:
QLoRaAccuracy = accuracyFunction(QLoRaModel, eval_data)

#IA3

In [ ]:
model_fp32 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
).to(device)

ia3_config = IA3Config(
    task_type="SEQ_CLS",
    inference_mode=False,
    target_modules=["query", "key", "value"],
    feedforward_modules=[],
    init_ia3_weights=True
)

model_ia3 = get_peft_model(model_fp32, ia3_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_args_IA3 = TrainingArguments(
    output_dir="ia3-bert-yelp",
    per_device_train_batch_size=16,
    learning_rate=5e-4,   # higher LR recommended for IA3
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none"
)

trainer_IA3 = Trainer(
    model=model_ia3,
    args=training_args_IA3,
    train_dataset=train_data,
    eval_dataset=eval_data
)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    memoryBeforeIA3 = torch.cuda.memory_allocated()
else:
    memoryBeforeIA3 = 0

In [ ]:
import time
start = time.time()
trainer_IA3.train()
end = time.time()

Epoch,Training Loss,Validation Loss
1,0.501800,0.469404


In [ ]:
IA3TrainingTime = end - start

In [ ]:
if torch.cuda.is_available():
    memoryAfterIA3 = torch.cuda.memory_allocated()
else:
    memoryAfterIA3 = 0

In [ ]:
IA3Accuracy = accuracyFunction(model_ia3, eval_data)

#Results

In [ ]:
print("GPU memory before LoRa:", memoryBeforeLora, "Bytes")
print("GPU memory After LoRa:", memoryAfterLoRa, "Bytes")
print("LoRa training time:", LoRaTrainingTime, "Seconds")
print("Lora Accuracy:", LoRaAccuracy)

GPU memory before LoRa:  1980083200 Bytes
GPU memory After LoRa:  1990888448 Bytes
LoRa training time: 548.0927684307098 Seconds
Lora Accuracy: 0.923


In [ ]:
print("GPU memory before QLoRa:", memoryBeforeQLora, "Bytes")
print("GPU memory After QLoRa:", memoryAfterQLoRa, "Bytes")
print("QLoRa training time:", QLoRaTrainingTime, "Seconds")
print("QLora Accuracy:", QLoRaAccuracy)

GPU memory before QLoRa: 2161966592 Bytes
GPU memory After QLoRa: 2183270912 Bytes
QLoRa training time: 390.0980987548828 Seconds
QLora Accuracy: 0.91125


In [ ]:
print("GPU memory before IA3:", memoryBeforeIA3, "Bytes")
print("GPU memory After IA3:", memoryAfterIA3, "Bytes")
print("IA3 training time:", IA3TrainingTime, "Seconds")
print("IA3 Accuracy:", IA3Accuracy)

GPU memory before IA3: 2624569856 Bytes
GPU memory After IA3: 2186155520 Bytes
IA3 training time: 102.93827056884766 Seconds
IA3 Accuracy: 0.79975
